In [1]:
import mne
import numpy as np
from pathlib import Path 
from scipy.io import loadmat
from mne.preprocessing import EOGRegression
from mne.decoding import CSP
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.metrics import cohen_kappa_score

In [2]:
np.random.seed(0)
mne.set_log_level('WARNING')

In [3]:
data_folder = Path(r'D:\Coding\BCI project\data')
subjects = ['A01', 'A02', 'A03', 'A04', 'A05', 'A06', 'A07', 'A08', 'A09']
ch_renamed = {
    'EEG-Fz':'Fz',
    'EEG-0':'FC3',
    'EEG-1':'FC1',
    'EEG-2':'FCz',
    'EEG-3':'FC2',
    'EEG-4':'FC4',
    'EEG-5':'C5',
    'EEG-C3':'C3',
    'EEG-6':'C1',
    'EEG-Cz':'Cz',
    'EEG-7':'C2',
    'EEG-C4':'C4',
    'EEG-8':'C6',
    'EEG-9':'CP3',
    'EEG-10':'CP1',
    'EEG-11':'CPz',
    'EEG-12':'CP2',
    'EEG-13':'CP4',
    'EEG-14':'P1',
    'EEG-Pz':'Pz',
    'EEG-15':'P2',
    'EEG-16':'POz',
}
motor_tasks = {1:'left hand vs rest', 2:'right hand vs rest', 3:'feet vs rest', 4:'tongue vs rest'}

In [4]:
def load_test_labels(subject, motor_tasks):
    labels_test_dict = {}
    mat_files = loadmat(Path(r'D:\Coding\BCI project\data\true labels') / f'{subject}E.mat')
    labels_test = mat_files['classlabel'].flatten()
    
    for task in motor_tasks:
        labels_binary = (labels_test == task).astype(int)
        labels_test_dict[task] = labels_binary
    return labels_test_dict, labels_test

In [5]:
def preprocessing(raw, ch_names):
    raw.rename_channels(ch_names)
    raw.set_channel_types({
        'EOG-left':'eog',
        'EOG-central':'eog',
        'EOG-right':'eog'
    })
    montage = mne.channels.make_standard_montage('standard_1005')
    raw.set_montage(montage, on_missing = 'ignore')
    raw.filter(l_freq = 1, h_freq = None, fir_design='firwin')
    raw.set_eeg_reference()
    return raw

In [6]:
def calibration_eog(raw):
    events, event_id = mne.events_from_annotations(raw)
    sfreq = raw.info['sfreq']
    
    def event_start_time(events, event_id, code):
        mask = (events[:, 2] == event_id[code])
        event_time = events[mask][0, 0] / sfreq
        return event_time

    run_start_time = event_start_time(events, event_id, '768')
    eye_move_time = event_start_time(events, event_id, '1072')

    if '276' not in event_id or '277' not in event_id:
        calibration = raw.copy().crop(eye_move_time, run_start_time)
        return calibration  
    else:
        eyes_open_time = event_start_time(events, event_id, '276')
        eyes_closed_time = event_start_time(events, event_id, '277')
        seg_open = raw.copy().crop(eyes_open_time, eyes_closed_time)
        seg_move = raw.copy().crop(eye_move_time, run_start_time)
        calibration = mne.concatenate_raws([seg_open, seg_move])
        return calibration

In [7]:
def regression_eog(raw, calibration):
    model_plain = EOGRegression(picks='eeg', picks_artifact='eog').fit(calibration)
    raw_clean_plain = model_plain.apply(raw)
    return raw_clean_plain

In [8]:
def filter_bank(raw):
    raw_dict = {}
    frequency_bands = [(4,8), (8, 12), (12, 16), (16, 20), (20, 24), (24, 28), (28, 32), (32, 36), (36, 40)]
    
    for l_freq, h_freq in frequency_bands:
        raw_dict[(l_freq, h_freq)] = raw.copy().filter(l_freq, h_freq, fir_design='firwin')
    return raw_dict

In [9]:
def epoching_train(raw_dict, motor_tasks):
    epochs_dict = {}
    epochs_data_dict = {}
    labels_dict = {}
    
    for band in raw_dict:    
        events, event_id = mne.events_from_annotations(raw_dict[band])
        classes = {'left hand': event_id['769'], 'right hand': event_id['770'], 
                 'feet': event_id['771'], 'tongue': event_id['772']}
        epochs =  mne.Epochs(raw_dict[band], events, event_id = classes, tmin = 0, 
                             tmax = 4, picks = 'eeg', baseline = None, preload = True)
        epochs_data = epochs.get_data(copy = False)
        
        labels = epochs.events[:, -1] - event_id['769'] + 1
        
        for task in motor_tasks:
            labels_binary = (labels == task).astype(int)
            labels_dict[task] = labels_binary
    
        epochs_dict[band] = epochs
        epochs_data_dict[band] = epochs_data
        
    return epochs_dict, epochs_data_dict, labels_dict

In [10]:
def epoching_test(raw_dict):
    epochs_dict = {}
    epochs_data_dict = {}

    for band in raw_dict:
        events, event_id = mne.events_from_annotations(raw_dict[band])
        epochs = mne.Epochs(raw_dict[band], events, event_id = {'unknown':event_id['783']}, tmin = 0,
                            tmax = 4, picks = 'eeg', baseline = None, preload = True)
        epochs_data = epochs.get_data(copy = False)

        epochs_dict[band] = epochs
        epochs_data_dict[band] = epochs_data
        
    return epochs_dict, epochs_data_dict

In [11]:
results = {}
class_balance = {1:[], 2:[], 3:[], 4:[]}
accuracy_dict = {}
true_labels_dict = {}
predicted_classes_dict = {}

for subject in subjects:
    
    train_files = data_folder / f'{subject}T.gdf' 
    test_files = data_folder / f'{subject}E.gdf'
    raw_train = mne.io.read_raw_gdf(train_files, preload = True)
    raw_test = mne.io.read_raw_gdf(test_files,  preload = True)

    labels_test_dict, true_labels = load_test_labels(subject, motor_tasks)

    raw_train = preprocessing(raw_train, ch_renamed)
    raw_test = preprocessing(raw_test, ch_renamed)

    calibration_train = calibration_eog(raw_train)
    calibration_test = calibration_eog(raw_test)

    raw_train = regression_eog(raw_train, calibration_train)
    raw_test = regression_eog(raw_test, calibration_test)

    raw_train_dict = filter_bank(raw_train)
    raw_test_dict = filter_bank(raw_test)

    epochs_train_dict, epochs_train_data_dict, labels_train_dict = epoching_train(raw_train_dict, motor_tasks)
    epochs_test_dict, epochs_test_data_dict = epoching_test(raw_test_dict)
    
    class_components = {}
    class_scores = {}
    results[subject] = {}
    
    for task in motor_tasks:
        X_train_list = []
        X_test_list = []
        csp_dict = {}
        
        lda = LinearDiscriminantAnalysis()
        selector = SelectKBest(score_func=mutual_info_classif, k = 14)
        
        for band in epochs_train_data_dict:
            csp = CSP(n_components = 4, reg='ledoit_wolf', log = True, norm_trace = False)
            csp_dict[band] = csp
            X_train = csp.fit_transform(epochs_train_data_dict[band], labels_train_dict[task])
            X_train_list.append(X_train)
        X_train_concatenated = np.concatenate(X_train_list, axis = 1)
        X_train_selected = selector.fit_transform(X_train_concatenated, labels_train_dict[task])
        
        for band in epochs_test_data_dict:
            X_test = csp_dict[band].transform(epochs_test_data_dict[band])
            X_test_list.append(X_test)
        X_test_concatenated = np.concatenate(X_test_list, axis = 1)
        X_test_selected = selector.transform(X_test_concatenated)
        
        lda.fit(X_train_selected, labels_train_dict[task])
        class_scores[task] = lda.decision_function(X_test_selected)

        class_components[task] = {'csp_dict':csp_dict, 'selector':selector, 'lda':lda}
        
        score = lda.score(X_test_selected, labels_test_dict[task])
        results[subject][task] = score
        class_balance[task].append(np.mean(labels_test_dict[task] == labels_test_dict[task][0]))

    scores_matrix = np.column_stack([class_scores[1], class_scores[2], class_scores[3], class_scores[4]])
    predicted_classes = np.argmax(scores_matrix, axis=1) + 1
    accuracy = np.mean(predicted_classes == true_labels)

    accuracy_dict[subject] = accuracy                    
    true_labels_dict[subject] = true_labels              
    predicted_classes_dict[subject] = predicted_classes   

C:\ProgramData\anaconda3\Lib\contextlib.py:148: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
C:\ProgramData\anaconda3\Lib\contextlib.py:148: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
C:\ProgramData\anaconda3\Lib\contextlib.py:148: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
C:\ProgramData\anaconda3\Lib\contextlib.py:148: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
C:\ProgramData\anaconda3\Lib\contextlib.py:148: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
C:\ProgramData\anaconda3\Lib\contextlib.py:148: RuntimeWarning: Channel names are not

In [12]:
print('=' * 100 )   
for task in motor_tasks:
    task_scores = []
    for subject in subjects:
        task_scores.append(results[subject][task])
    final_score = np.mean(task_scores)
    final_std = np.std(task_scores)
    class_balance_mean = np.mean(class_balance[task])
    chance_level = max(class_balance_mean, 1 - class_balance_mean)
    
    print('=' * 100 )
    print(f'Total classification precision for {motor_tasks[task]} is {final_score * 100}%')
    print(f'Standard deviation for {motor_tasks[task]} is {final_std}')
    print(f'Chance level for {motor_tasks[task]} is {chance_level * 100}%')
    for sbjct in results.keys():
        print(f'subject {sbjct} - {motor_tasks[task]} score {results[sbjct][task]}')
    print('=' * 100 )
print('=' * 100 )

Total classification precision for left hand vs rest is 83.68055555555556%
Standard deviation for left hand vs rest is 0.06282067937551811
Chance level for left hand vs rest is 69.44444444444444%
subject A01 - left hand vs rest score 0.8923611111111112
subject A02 - left hand vs rest score 0.75
subject A03 - left hand vs rest score 0.875
subject A04 - left hand vs rest score 0.7916666666666666
subject A05 - left hand vs rest score 0.8402777777777778
subject A06 - left hand vs rest score 0.7708333333333334
subject A07 - left hand vs rest score 0.8854166666666666
subject A08 - left hand vs rest score 0.78125
subject A09 - left hand vs rest score 0.9444444444444444
Total classification precision for right hand vs rest is 78.66512345679013%
Standard deviation for right hand vs rest is 0.10385899922496941
Chance level for right hand vs rest is 75.0%
subject A01 - right hand vs rest score 0.8402777777777778
subject A02 - right hand vs rest score 0.7569444444444444
subject A03 - right hand vs

In [13]:
kappa_dict = {}
for subject in subjects:
    kappa_dict[subject] = cohen_kappa_score(true_labels_dict[subject], predicted_classes_dict[subject])

final_kappa = np.mean(list(kappa_dict.values()))
kappa_std = np.std(list(kappa_dict.values()))

overall_accuracy = np.mean(list(accuracy_dict.values()))
overall_accuracy_std = np.std(list(accuracy_dict.values()))

print(f'Mean 4-class accuracy: {overall_accuracy*100:.2f}% ± {overall_accuracy_std*100:.2f}%')
print(f'Mean kappa: {final_kappa:.3f} ± {kappa_std:.3f}')
for subject in subjects:
    print(f'subject {subject} - accuracy {accuracy_dict[subject]*100:.2f}% - kappa {kappa_dict[subject]:.3f}')

Mean 4-class accuracy: 67.79% ± 12.75%
Mean kappa: 0.570 ± 0.170
subject A01 - accuracy 79.51% - kappa 0.727
subject A02 - accuracy 52.43% - kappa 0.366
subject A03 - accuracy 79.86% - kappa 0.731
subject A04 - accuracy 69.79% - kappa 0.597
subject A05 - accuracy 56.25% - kappa 0.417
subject A06 - accuracy 44.79% - kappa 0.264
subject A07 - accuracy 78.47% - kappa 0.713
subject A08 - accuracy 68.40% - kappa 0.579
subject A09 - accuracy 80.56% - kappa 0.741
